In [ ]:
pip install transformers torch

In [1]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import precision_score, recall_score, f1_score

# -------------------------------
# 1. LOAD DATASET
# -------------------------------
test_path = "../Datasets/NYT-10 dataset/test.json"

data = []
with open(test_path, "r") as f:
    for line in f:
        data.append(json.loads(line))

print("Total samples:", len(data))
print("Sample:", data[0])

c:\Users\CT-PROJECT\Documents\Team10_FYP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total samples: 5000
Sample: {'sentText': 'But that spasm of irritation by a master intimidator was minor compared with what Bobby Fischer , the erratic former world chess champion , dished out in March at a news conference in Reykjavik , Iceland .', 'articleId': '/m/vinci8/data1/riedel/projects/relation/kb/nyt1/docstore/nyt-2005-2006.backup/1677367.xml.pb', 'relationMentions': [{'em1Text': 'Bobby Fischer', 'em2Text': 'Iceland', 'label': '/people/person/nationality'}, {'em1Text': 'Iceland', 'em2Text': 'Reykjavik', 'label': '/location/country/capital'}, {'em1Text': 'Iceland', 'em2Text': 'Reykjavik', 'label': '/location/location/contains'}, {'em1Text': 'Bobby Fischer', 'em2Text': 'Reykjavik', 'label': '/people/deceased_person/place_of_death'}], 'entityMentions': [{'start': 0, 'label': 'PERSON', 'text': 'Bobby Fischer'}, {'start': 1, 'label': 'LOCATION', 'text': 'Reykjavik'}, {'start': 2, 'label': 'LOCATION', 'text': 'Iceland'}], 'sentId': '1'}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# -------------------------------
# 2. PREPARE SAMPLES (GT entities)
# -------------------------------
samples = []

for item in data[:20]:   # keep small first (faster testing)

    sentence = item["sentText"]

    if "relationMentions" in item and len(item["relationMentions"]) > 0:
        for rm in item["relationMentions"]:
            samples.append({
                "sentence": sentence,
                "head": rm["em1Text"].lower(),
                "tail": rm["em2Text"].lower()
            })
    else:
        # fallback (no relation case)
        entity_mentions = item.get("entityMentions", [])

        head = "unknown"
        tail = "unknown"
        if len(entity_mentions) >= 2:
            head = entity_mentions[0]["text"].lower()
            tail = entity_mentions[1]["text"].lower()

        samples.append({
            "sentence": sentence,
            "head": head,
            "tail": tail
        })

print("Total processed samples:", len(samples))

Total processed samples: 40


In [12]:
data

[{'sentText': 'But that spasm of irritation by a master intimidator was minor compared with what Bobby Fischer , the erratic former world chess champion , dished out in March at a news conference in Reykjavik , Iceland .',
  'articleId': '/m/vinci8/data1/riedel/projects/relation/kb/nyt1/docstore/nyt-2005-2006.backup/1677367.xml.pb',
  'relationMentions': [{'em1Text': 'Bobby Fischer',
    'em2Text': 'Iceland',
    'label': '/people/person/nationality'},
   {'em1Text': 'Iceland',
    'em2Text': 'Reykjavik',
    'label': '/location/country/capital'},
   {'em1Text': 'Iceland',
    'em2Text': 'Reykjavik',
    'label': '/location/location/contains'},
   {'em1Text': 'Bobby Fischer',
    'em2Text': 'Reykjavik',
    'label': '/people/deceased_person/place_of_death'}],
  'entityMentions': [{'start': 0, 'label': 'PERSON', 'text': 'Bobby Fischer'},
   {'start': 1, 'label': 'LOCATION', 'text': 'Reykjavik'},
   {'start': 2, 'label': 'LOCATION', 'text': 'Iceland'}],
  'sentId': '1'},
 {'sentText': "B

In [3]:
# -------------------------------
# 3. LOAD LLAMA MODEL
# -------------------------------
model_name = "meta-llama/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

c:\Users\CT-PROJECT\Documents\Team10_FYP\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\CT-PROJECT\.cache\huggingface\hub\models--meta-llama--Llama-2-7b-chat-hf. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
W0318 10:00:38.599000 19432 Lib\site-

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
   

In [ ]:
# -------------------------------
# 4. PROMPT FUNCTION
# -------------------------------
def create_prompt_relation(sentence):
    return f"""Select the TWO entities in the sentence that are most likely related.

Rules:
- Choose the pair that has a meaningful relationship
- Prefer named entities (people, locations, organizations)
- Do NOT return generic words
- Do NOT return placeholders

Sentence: {sentence}

Return ONLY in this format (without quotes):
Head: <entity>
Tail: <entity>

Answer:
"""

### Entity matching between model and ground truth samples

In [75]:
def create_prompt(sentence):
    return f"""
Extract all named entities from the sentence.

Rules:
- Only proper nouns (person, location, organization)
- No generic words
- No explanations

Sentence: {sentence}

Entities:
"""

In [84]:
def extract_all_entities(output_text):
    output_text = output_text.lower()
    print("Raw model output:", output_text)
    # Step 1: isolate only the entity section
    if "entities:" in output_text:
        output_text = output_text.split("entities:")[-1]

    # Step 2: split line by line
    lines = output_text.strip().split("\n")

    entities = []
    for line in lines:
        line = line.strip()

        # stop if model starts generating something else
        if "sentence:" in line:
            break

        # remove bullets if any
        line = line.replace("-", "").strip()

        # filters
        if len(line) < 3:
            continue
        if any(x in line for x in ["rules", "explanations"]):
            continue

        entities.append(line)

    return list(set(entities))

In [77]:
def get_gt_entities(item):
    entities = set()

    # from relationMentions
    for rm in item.get("relationMentions", []):
        entities.add(rm["em1Text"].lower())
        entities.add(rm["em2Text"].lower())

    # from entityMentions
    for em in item.get("entityMentions", []):
        entities.add(em["text"].lower())

    return list(entities)

In [85]:
def clean_pred_entities(entities):
    banned = {"<entity1>", "<entity2>", "entity1", "entity2"}

    cleaned = []
    for e in entities:
        if e in banned:
            continue
        if len(e) < 3:   # remove very short junk
            continue
        cleaned.append(e)

    return cleaned

In [43]:
def is_match(true_head, true_tail, pred_entities):
    return (
        true_head in pred_entities and
        true_tail in pred_entities
    )

In [86]:
def score_sentence(gt_entities, pred_entities):
    gt_set = set(gt_entities)
    pred_set = set(pred_entities)

    intersection = gt_set & pred_set

    precision = len(intersection) / len(pred_set) if pred_set else 0
    recall = len(intersection) / len(gt_set) if gt_set else 0

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1

In [ ]:
def predict_entities(sentence,tokenizer, model):
    prompt = create_prompt(sentence)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.1,
            do_sample=False
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    pred_entities = extract_all_entities(text)
    pred_entities = clean_pred_entities(pred_entities)

    return pred_entities

In [88]:
def evaluate(data, tokenizer, model):
    total_p, total_r, total_f1 = 0, 0, 0

    for item in data[:50]:   # increase later
        sentence = item["sentText"]

        gt_entities = get_gt_entities(item)
        pred_entities = predict_entities(sentence, tokenizer, model)

        p, r, f1 = score_sentence(gt_entities, pred_entities)

        total_p += p
        total_r += r
        total_f1 += f1

        print("\nSentence:", sentence)
        print("GT   :", gt_entities)
        print("Pred :", pred_entities)
        print(f"P={p:.2f}, R={r:.2f}, F1={f1:.2f}")

    n = len(data[:50])

    print("\n📊 FINAL SCORES")
    print("Precision:", total_p / n)
    print("Recall   :", total_r / n)
    print("F1 Score :", total_f1 / n)

Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw model output: 
extract all named entities from the sentence.

rules:
- only proper nouns (person, location, organization)
- no generic words
- no explanations

sentence: but that spasm of irritation by a master intimidator was minor compared with what bobby fischer , the erratic former world chess champion , dished out in march at a news conference in reykjavik , iceland .

entities:
bobby fischer
reykjavik
iceland

Sentence: But that spasm of irritation by a master intimidator was minor compared with what Bobby Fischer , the erratic former world chess champion , dished out in March at a news conference in Reykjavik , Iceland .
GT   : ['iceland', 'bobby fischer', 'reykjavik']
Pred : ['iceland', 'bobby fischer', 'reykjavik']
P=1.00, R=1.00, F1=1.00


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw model output: 
extract all named entities from the sentence.

rules:
- only proper nouns (person, location, organization)
- no generic words
- no explanations

sentence: but schaap seems as comfortable in that role as joe buck , the fox baseball and football sportscaster who so clearly benefited from learning beside his father , jack buck , the late voice of the st. louis cardinals . ''

entities:
schaap
joe buck
fox
st. louis cardinals
jack buck

Sentence: But Schaap seems as comfortable in that role as Joe Buck , the Fox baseball and football sportscaster who so clearly benefited from learning beside his father , Jack Buck , the late voice of the St. Louis Cardinals . ''
GT   : ['st. louis cardinals', 'joe buck', 'fox', 'jack buck']
Pred : ['schaap', 'joe buck', 'st. louis cardinals', 'fox', 'jack buck']
P=0.80, R=1.00, F1=0.89


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Raw model output: 
extract all named entities from the sentence.

rules:
- only proper nouns (person, location, organization)
- no generic words
- no explanations

sentence: over the last few years , as its residents fought a battle against the city to keep their homes , the fort trumbull neighborhood of new london looked more and more like a war-torn village , with acres of vacant lots interrupted intermittently by debris and a few scattered houses .

entities:
fort trumbull
new london

Sentence: OVER the last few years , as its residents fought a battle against the city to keep their homes , the Fort Trumbull neighborhood of New London looked more and more like a war-torn village , with acres of vacant lots interrupted intermittently by debris and a few scattered houses .
GT   : ['new london', 'fort trumbull']
Pred : ['new london', 'fort trumbull']
P=1.00, R=1.00, F1=1.00


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw model output: 
extract all named entities from the sentence.

rules:
- only proper nouns (person, location, organization)
- no generic words
- no explanations

sentence: curry and crawford , close friends and longtime teammates in chicago and new york , were in sync in the final quarter .

entities:
curry
crawford
chicago
new york

Sentence: Curry and Crawford , close friends and longtime teammates in Chicago and New York , were in sync in the final quarter .
GT   : ['crawford', 'curry', 'new york', 'chicago']
Pred : ['crawford', 'new york', 'curry', 'chicago']
P=1.00, R=1.00, F1=1.00


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw model output: 
extract all named entities from the sentence.

rules:
- only proper nouns (person, location, organization)
- no generic words
- no explanations

sentence: the knicks were 10 for 15 from the free-throw line in the fourth quarter , but had only 2 turnovers , and coach isiah thomas called it a vast improvement from their fourth quarter in chicago . ''

entities:

1. knicks
2. chicago
3. isiah thomas

Sentence: The Knicks were 10 for 15 from the free-throw line in the fourth quarter , but had only 2 turnovers , and Coach Isiah Thomas called it a vast improvement from their fourth quarter in Chicago . ''
GT   : ['isiah thomas', 'chicago']
Pred : ['3. isiah thomas', '2. chicago', '1. knicks']
P=0.00, R=0.00, F1=0.00
Raw model output: 
extract all named entities from the sentence.

rules:
- only proper nouns (person, location, organization)
- no generic words
- no explanations

sentence: nanette lepore , a new york fashion designer , said she reserved a seat every week to g

In [40]:
# -------------------------------
# 5. EXTRACT ENTITIES FROM OUTPUT
# -------------------------------
def extract_entities(output_text):

    output_text = output_text.lower()

    head = "unknown"
    tail = "unknown"

    for line in output_text.split("\n"):
        if "head:" in line:
            head = line.split("head:")[-1].strip()
        elif "tail:" in line:
            tail = line.split("tail:")[-1].strip()

    return head, tail

In [ ]:
def extract_all_entities_relation(output_text):
    output_text = output_text.lower()

    entities = []

    for line in output_text.split("\n"):
        if "head:" in line or "tail:" in line:
            parts = line.split(":")[-1]

            # split multiple entities
            candidates = parts.split(",")

            for c in candidates:
                c = c.strip()
                if c:
                    entities.append(c)

    return list(set(entities))  # remove duplicates

In [44]:
correct = 0

for i in range(len(samples)):
    sample = samples[i]

    true_head = sample["head"]
    true_tail = sample["tail"]

    pred_entities = extract_all_entities(
        tokenizer.decode(
            model.generate(
                **tokenizer(create_prompt(sample["sentence"]), return_tensors="pt").to(model.device),
                max_new_tokens=50,
                temperature=0.1,
                do_sample=False
            )[0],
            skip_special_tokens=True
        )
    )
    pred_entities = clean_pred_entities(pred_entities)
    print(f"\nSentence: {sample['sentence']}")
    print(f"GT   : {(true_head, true_tail)}")
    print(f"Pred : {pred_entities}")
    if is_match(true_head, true_tail, pred_entities):
        correct += 1
    

accuracy = correct / len(samples)
print("Relaxed Accuracy:", accuracy)

Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sentence: But that spasm of irritation by a master intimidator was minor compared with what Bobby Fischer , the erratic former world chess champion , dished out in March at a news conference in Reykjavik , Iceland .
GT   : ('bobby fischer', 'iceland')
Pred : ['<entity>', 'bobby fischer', 'reykjavik']


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sentence: But that spasm of irritation by a master intimidator was minor compared with what Bobby Fischer , the erratic former world chess champion , dished out in March at a news conference in Reykjavik , Iceland .
GT   : ('iceland', 'reykjavik')
Pred : ['<entity>', 'bobby fischer', 'reykjavik']


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Sentence: As a half Englishman , half Frenchman who arrived in New York from Paris in 1965 at 23 , and has opened and operated fashion boutiques , night clubs and now restaurants , including La Goulue on Madison Avenue , Mr. Denoyer has served the moneyed men and women of Manhattan well , putting them at ease on their own terms .
GT   : ('la goulue', 'paris')
Pred : ['<entity>', 'la goulue', 'denoyer']


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sentence: The couple have two daughters , Seon-yong , 34 , who works in Seoul for the Korea Foundation , a nonprofit organization that promotes Korean culture , and Hyun-hee , 30 , a field officer for Unicef in Nairobi , Kenya .
GT   : ('kenya', 'nairobi')
Pred : ['couple', '<entity>', 'daughters']


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sentence: The couple have two daughters , Seon-yong , 34 , who works in Seoul for the Korea Foundation , a nonprofit organization that promotes Korean culture , and Hyun-hee , 30 , a field officer for Unicef in Nairobi , Kenya .
GT   : ('kenya', 'nairobi')
Pred : ['couple', '<entity>', 'daughters']

Sentence: WHERE -- Dixie , Idaho WHAT -- 2-bedroom house HOW MUCH $ 285,000 This 1,400-square-foot log house is on 10 acres bordered on three sides by the Nez Perce National Forest .
GT   : ('idaho', 'nez perce national forest')
Pred : ['<entity>', 'dixie', 'idaho']
Relaxed Accuracy: 0.175


In [6]:
# -------------------------------
# 6. PREDICT FUNCTION
# -------------------------------
def predict_entities(sentence):

    prompt = create_prompt(sentence)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.1,
            do_sample=False
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return extract_entities(text)

In [7]:
len(samples)

40

In [9]:
# -------------------------------
# 7. RUN PREDICTIONS
# -------------------------------
y_true = []
y_pred = []
def normalize_pair(h, t):
    return tuple(sorted([h.strip(), t.strip()]))


for i in range(len(samples)):
    sample=samples[i]
    sentence = sample["sentence"]

    true_head = sample["head"]
    true_tail = sample["tail"]

    pred_head, pred_tail = predict_entities(sentence)

    y_true.append(normalize_pair(true_head, true_tail))
    y_pred.append(normalize_pair(pred_head, pred_tail))

    print(f"\nSentence: {sentence}")
    print(f"GT   : {normalize_pair(true_head, true_tail)}")
    print(f"Pred : {normalize_pair(pred_head, pred_tail)}")

Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sentence: But that spasm of irritation by a master intimidator was minor compared with what Bobby Fischer , the erratic former world chess champion , dished out in March at a news conference in Reykjavik , Iceland .
GT   : ('bobby fischer', 'iceland')
Pred : ('bobby fischer', 'iceland')


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sentence: But that spasm of irritation by a master intimidator was minor compared with what Bobby Fischer , the erratic former world chess champion , dished out in March at a news conference in Reykjavik , Iceland .
GT   : ('iceland', 'reykjavik')
Pred : ('bobby fischer', 'iceland')


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Sentence: As a half Englishman , half Frenchman who arrived in New York from Paris in 1965 at 23 , and has opened and operated fashion boutiques , night clubs and now restaurants , including La Goulue on Madison Avenue , Mr. Denoyer has served the moneyed men and women of Manhattan well , putting them at ease on their own terms .
GT   : ('la goulue', 'paris')
Pred : ('englishman', 'frenchman')


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sentence: The couple have two daughters , Seon-yong , 34 , who works in Seoul for the Korea Foundation , a nonprofit organization that promotes Korean culture , and Hyun-hee , 30 , a field officer for Unicef in Nairobi , Kenya .
GT   : ('kenya', 'nairobi')
Pred : ('couple', 'daughters')


Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sentence: The couple have two daughters , Seon-yong , 34 , who works in Seoul for the Korea Foundation , a nonprofit organization that promotes Korean culture , and Hyun-hee , 30 , a field officer for Unicef in Nairobi , Kenya .
GT   : ('kenya', 'nairobi')
Pred : ('couple', 'daughters')

Sentence: WHERE -- Dixie , Idaho WHAT -- 2-bedroom house HOW MUCH $ 285,000 This 1,400-square-foot log house is on 10 acres bordered on three sides by the Nez Perce National Forest .
GT   : ('idaho', 'nez perce national forest')
Pred : ('dixie', 'idaho')


In [14]:
correct = 0

for t, p in zip(y_true, y_pred):
    if t == p or t[0]==p[1] or t[1]==p[0] or t[0]==p[0] or t[1]==p[1]:
        correct += 1

accuracy = correct / len(y_true)
print("Accuracy:", accuracy)

Accuracy: 0.4


In [15]:
def partial_match_score(true_pair, pred_pair):
    return len(set(true_pair) & set(pred_pair)) / 2

scores = [partial_match_score(t, p) for t, p in zip(y_true, y_pred)]
print("Partial Match Score:", sum(scores)/len(scores))

Partial Match Score: 0.2375


In [20]:
# -------------------------------
# 8. CONVERT FOR METRICS
# -------------------------------
y_true_str = [f"{h}|{t}" for h, t in y_true]
y_pred_str = [f"{h}|{t}" for h, t in y_pred]


# -------------------------------
# 9. METRICS
# -------------------------------
precision = precision_score(y_true_str, y_pred_str, average="macro", zero_division=0)
recall = recall_score(y_true_str, y_pred_str, average="macro", zero_division=0)
f1 = f1_score(y_true_str, y_pred_str, average="macro", zero_division=0)

print("\n📊 Entity Extraction Results")
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)



📊 Entity Extraction Results
Precision: 0.005434782608695652
Recall: 0.021739130434782608
F1 Score: 0.008695652173913044


In [11]:
# -------------------------------
# 10. STRICT MATCH (OPTIONAL)
# -------------------------------
correct = sum([1 for gt, pred in zip(y_true, y_pred) if gt == pred])

strict_precision = correct / len(y_pred)
strict_recall = correct / len(y_true)
strict_f1 = (2 * strict_precision * strict_recall) / (strict_precision + strict_recall + 1e-8)

print("\n📊 Strict Match Results")
print("Precision:", strict_precision)
print("Recall:", strict_recall)
print("F1:", strict_f1)


📊 Strict Match Results
Precision: 0.025
Recall: 0.025
F1: 0.024999995000001003
